# AWS Bedrock + LangChain Document Q&A

## Retrieval-Augmented Generation (RAG) Pipeline

This notebook implements a document question-answering workflow using **Amazon S3, Amazon SageMaker, Amazon Bedrock, LangChain, Amazon Titan Embeddings, FAISS, and Amazon Nova Lite**.

The pipeline:

> **PDF → S3 → PDF loading → text chunking → embeddings → FAISS vector store → similarity retrieval → Amazon Nova Lite → answer**

### Learning objectives

By the end of this notebook, you should understand:

- how a PDF is converted into searchable text chunks;
- how text embeddings represent document meaning as vectors;
- how FAISS performs similarity-based retrieval;
- how retrieved context is supplied to a foundation model;
- how these components combine to form a basic **RAG** system.

> **Portfolio note:** This notebook is intentionally written as a technical demonstration. It does not contain AWS credentials. Use an IAM role/environment that already has permission to access S3 and Amazon Bedrock.


## 1. Install dependencies

The original lab used a SageMaker `conda_python3` environment. The packages below are the libraries required by the notebook.

**Important:** package versions can change over time. The pins below reflect the working lab environment and are intended to make the notebook reproducible there.


In [ ]:
!pip install --upgrade pip
!pip install --upgrade "numpy<2"
!pip install --upgrade langchain
!pip install --upgrade langchain-community
!pip install --upgrade langchain-text-splitters
!pip install --upgrade langchain-aws
!pip install --upgrade faiss-cpu==1.7.4
!pip install --upgrade pypdf
!pip install --upgrade boto3


## 2. Import libraries

The main components are:

- **boto3** — communicates with AWS services such as S3.
- **PyPDFLoader** — extracts text from the PDF.
- **RecursiveCharacterTextSplitter** — breaks the document into overlapping chunks.
- **BedrockEmbeddings** — creates vector embeddings using Amazon Titan.
- **FAISS** — stores vectors and performs similarity search.
- **ChatBedrock** — sends the retrieved context to an Amazon Bedrock chat model.


In [ ]:
import boto3

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_aws import BedrockEmbeddings, ChatBedrock


## 3. Configure AWS and document settings

Set the values for your own environment.

The notebook deliberately uses **placeholders** rather than embedding lab-specific account details or credentials.

If you run this in SageMaker with an attached IAM role, `boto3` can use the role's temporary credentials automatically.


In [ ]:
AWS_REGION = "us-east-1"

# Replace this with the S3 bucket that contains your PDF.
S3_BUCKET = "YOUR_S3_BUCKET_NAME"

# S3 object key and local notebook filename.
S3_KEY = "document.pdf"
LOCAL_PDF = "document.pdf"

# Retrieval configuration.
TOP_K = 3


## 4. Download the PDF from Amazon S3

The S3 object is downloaded into the notebook's local environment so that LangChain can process it.


In [ ]:
s3 = boto3.client("s3", region_name=AWS_REGION)

def read_s3_pdf(bucket: str, key: str, local_file: str = LOCAL_PDF) -> str:
    """Download a PDF from Amazon S3 and return the local file path."""
    if not bucket or bucket == "YOUR_S3_BUCKET_NAME":
        raise ValueError(
            "Set S3_BUCKET in section 3 to the name of the bucket that holds your PDF."
        )
    s3.download_file(bucket, key, local_file)
    return local_file

# Run this after replacing S3_BUCKET with your own bucket name.
local_path = read_s3_pdf(S3_BUCKET, S3_KEY, LOCAL_PDF)
print(f"Downloaded file to: {local_path}")


## 5. Load the PDF and split it into chunks

Large documents are difficult to pass to an LLM as one block of text. RAG systems therefore split them into smaller pieces.

Here:

- `chunk_size=500` controls the approximate chunk size.
- `chunk_overlap=50` keeps some text from the previous chunk so that important context is not lost at chunk boundaries.


In [ ]:
loader = PyPDFLoader(LOCAL_PDF)
pages = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = splitter.split_documents(pages)

print(f"Loaded {len(pages)} page(s).")
print(f"Created {len(docs)} document chunks.")


## 6. Create semantic embeddings with Amazon Titan

An embedding model converts text into a numerical vector that captures semantic meaning.

Conceptually:

```text
"document text"
      ↓
Titan Embeddings
      ↓
[0.12, -0.41, 0.87, ...]
```

These vectors allow the system to compare the meaning of a user query with the meaning of document chunks.


In [ ]:
embeddings = BedrockEmbeddings(
    model_id="amazon.titan-embed-text-v2:0",
    region_name=AWS_REGION
)

print("Embedding model initialized.")


## 7. Build the FAISS vector store

Each document chunk is embedded and stored in a FAISS vector index.

FAISS is used here for efficient similarity search over the document vectors.


In [ ]:
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

print("Vector store created successfully.")


## 8. Create the retriever

The retriever searches the vector store and returns the most relevant chunks for a question.

`k=3` means the pipeline retrieves the top three matching chunks.


In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)

print(f"Retriever created successfully (top_k={TOP_K}).")


## 9. Test document retrieval

Before involving the LLM, test the retrieval layer by asking a question and inspecting the chunks that FAISS returns.

This is an important RAG debugging step: if retrieval returns irrelevant context, the generation stage cannot reliably fix the problem.


In [ ]:
query = "What is Lorem Ipsum?"

retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} relevant chunk(s):\n")

for i, doc in enumerate(retrieved_docs, start=1):
    page_number = doc.metadata.get("page")
    page_label = f"Page {page_number + 1}" if isinstance(page_number, int) else "Page unknown"

    print(f"--- Chunk {i} ({page_label}) ---")
    print(doc.page_content)
    print()


## 10. Initialize the Amazon Bedrock foundation model

The retrieval system finds relevant information. The foundation model then uses that retrieved context to generate a natural-language answer.

This notebook uses **Amazon Nova Lite** through Amazon Bedrock.


In [ ]:
llm = ChatBedrock(
    model_id="amazon.nova-lite-v1:0",
    region_name=AWS_REGION
)

print("Amazon Nova Lite initialized.")


## 11. Generate an answer using retrieved context

This is the core RAG step:

1. retrieve relevant chunks;
2. combine those chunks into context;
3. place the context and question into a prompt;
4. ask Amazon Nova Lite to answer using the provided context.


In [ ]:
def answer_question(query: str) -> str:
    """Retrieve relevant document context and generate an answer with Bedrock."""
    retrieved_docs = retriever.invoke(query)

    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    prompt = f"""
Answer the question using only the context provided below.
If the answer cannot be found in the context, say that the document
does not provide enough information.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)
    return response.content

query = "What is Lorem Ipsum?"
answer = answer_question(query)

print("Question:")
print(query)
print("\nAnswer:")
print(answer)


## 12. Try additional questions

The same retrieval-and-generation pipeline can answer multiple questions about the loaded document.

Change the question below and run the cell again.


In [ ]:
query = "Replace this with a question about your PDF"

answer = answer_question(query)

print("Question:")
print(query)
print("\nAnswer:")
print(answer)


## 13. RAG pipeline summary

The complete flow implemented in this notebook is:

```text
                PDF Document
                      │
                      ▼
                 Amazon S3
                      │
                      ▼
              SageMaker / Jupyter
                      │
                      ▼
                 PDF Loader
                      │
                      ▼
                Text Chunks
                      │
                      ▼
           Amazon Titan Embeddings
                      │
                      ▼
                FAISS Vector Store
                      │
                User Question
                      │
                      ▼
                  Retriever
                      │
                      ▼
             Relevant Context
                      │
                      ▼
              Amazon Nova Lite
                      │
                      ▼
                 Final Answer
```

### Key distinction

- **Titan Embeddings** → converts text into vectors for semantic retrieval.
- **FAISS** → stores/searches those vectors.
- **Nova Lite** → generates the final natural-language answer.
- **LangChain** → provides the components that connect the pipeline.

### Security

Never hard-code AWS access keys, secret keys, passwords, or session tokens into this notebook or a public GitHub repository. Prefer IAM roles, environment-based credentials, or other AWS-supported credential providers.
